Copyright 2026 DataRobot, Inc. and its affiliates.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

In [ ]:
from jointfm_client import bootstrap_notebook

bootstrap_notebook(add_src_root=True)

# Feature Importance
Which of three market inputs (equity index level, 10-year Treasury yield, and one EUR/USD FX rate) drives the forecast of USD portfolio NAV and realized volatility over the next 10 ordinal horizons? The history is the same 100 positive float daily observations the other forecast notebooks use.

`client.feature_importance(...)` answers with permutation importance: it shuffles one feature column across the history rows, which keeps the column's values but destroys their order in time, forecasts again, and compares that sample forecast with the one from the untouched history. A feature the model relies on moves the forecast when shuffled; a feature it ignores leaves it in place. Each feature gets two scores per target and horizon:

- `mean`: the absolute shift of the forecast mean, in the target's units.
- `distance`: the centered squared 2-Wasserstein distance between the two sample sets, which pairs the sorted draws of both sets and averages their squared gaps (halved). Both sets are centered first, so it catches a change in spread, skew, or tails that leaves the mean in place. Its unit is the target's unit squared.

The call sends one sample forecast for the baseline plus one per feature, all at the same `n_samples` and `seed`, and returns one entry per feature. The cell below flattens the entries into one row per feature, target, and horizon.

In [ ]:
from pathlib import Path

import pandas as pd

from jointfm_client import JointFMClient, plan_forecast_columns

HISTORY_PATH = Path("notebooks/history.csv")
FEATURE_COLUMNS = ["equity_index_level", "treasury_10y_yield", "eur_usd_rate"]
TARGET_COLUMNS = ["portfolio_nav", "realized_volatility"]
N_SAMPLES = 1024
INPUT_STEPS = 100
OUTPUT_HORIZONS = 10
EXPECTED_COLUMNS = FEATURE_COLUMNS + TARGET_COLUMNS
QUERY_TIMES = list(range(INPUT_STEPS, INPUT_STEPS + OUTPUT_HORIZONS))
HORIZONS = list(range(1, OUTPUT_HORIZONS + 1))

history = pd.read_csv(HISTORY_PATH, dtype=float)
if list(history.columns) != EXPECTED_COLUMNS:
    raise ValueError(
        f"Expected columns {EXPECTED_COLUMNS!r}, got {list(history.columns)!r}"
    )
if len(history) != INPUT_STEPS:
    raise ValueError(f"Expected {INPUT_STEPS} history rows, got {len(history)}")

client = JointFMClient.from_env()
instance_ids = [
    entry.instance_id for entry in client.health_instances(cache=True).instances
]
print(f"JointFM instance(s): {instance_ids}")
plan = plan_forecast_columns(
    health=client.health(cache=True),
    feature_columns=FEATURE_COLUMNS,
    target_columns=TARGET_COLUMNS,
    history_length=len(history),
    query_times_length=len(QUERY_TIMES),
)
importance = client.feature_importance(
    history,
    query_times=QUERY_TIMES,
    horizons=HORIZONS,
    feature_columns=FEATURE_COLUMNS,
    target_columns=TARGET_COLUMNS,
    columns=plan.columns,
    n_samples=N_SAMPLES,
    seed=7,
)


def importance_scores(entries):
    """Return one row per feature, target, and horizon."""
    return pd.DataFrame(
        [
            {
                "feature": entry["feature"],
                "target": target,
                "horizon": horizon,
                "mean": entry["mean"][target][horizon],
                "distance": entry["distance"][target][horizon],
            }
            for entry in entries
            for target in TARGET_COLUMNS
            for horizon in HORIZONS
        ]
    )


scores = importance_scores(importance)
expected_score_rows = len(FEATURE_COLUMNS) * len(TARGET_COLUMNS) * OUTPUT_HORIZONS
if len(scores) != expected_score_rows:
    raise ValueError(f"Expected {expected_score_rows} score rows, got {len(scores)}")
scores

## Ranking the features
Both scores carry their target's units, so features compare within a target, never across targets. Averaging each score over the horizons gives one number per feature and target, and sorting by it ranks the features for each target. The chart below draws the mean-shift ranking with one panel per target, each on its own scale, top feature at the top.

Two limits hold for any ranking:

- Importance is measured against everything else the model still sees: the targets' own history and the other features stay in place. Two features that carry the same information can therefore both score low, because shuffling one leaves the other.
- A high score says the forecast depends on the feature's history, not that the feature causes the target.

In [ ]:
def rank_features(scores):
    """Average each score over the horizons and rank features per target."""
    return (
        scores.groupby(["target", "feature"])[["mean", "distance"]]
        .mean()
        .sort_values(["target", "mean"], ascending=[True, False])
    )


ranking = rank_features(scores)
ranking

In [ ]:
import matplotlib.pyplot as plt

SURFACE = "#fcfcfb"
BAR_COLOR = "#2a78d6"
TEXT_PRIMARY = "#0b0b0b"
TEXT_SECONDARY = "#52514e"
TEXT_MUTED = "#898781"
GRIDLINE = "#e1e0d9"
BASELINE = "#c3c2b7"
BAR_HEIGHT = 0.4
PANEL_WIDTH_INCHES = 5.0
PANEL_HEIGHT_INCHES = 2.6

figure, axes = plt.subplots(
    1,
    len(TARGET_COLUMNS),
    figsize=(PANEL_WIDTH_INCHES * len(TARGET_COLUMNS), PANEL_HEIGHT_INCHES),
    squeeze=False,
    layout="constrained",
    facecolor=SURFACE,
)
for axis, target in zip(axes[0], TARGET_COLUMNS, strict=True):
    shifts = ranking.loc[target, "mean"].sort_values()
    axis.barh(shifts.index, shifts.to_numpy(), height=BAR_HEIGHT, color=BAR_COLOR)
    axis.set_facecolor(SURFACE)
    axis.set_title(target, loc="left", color=TEXT_PRIMARY)
    axis.set_xlabel("mean shift in target units", color=TEXT_MUTED)
    axis.ticklabel_format(axis="x", style="sci", scilimits=(-3, 4), useMathText=True)
    axis.xaxis.get_offset_text().set_color(TEXT_MUTED)
    axis.grid(axis="x", color=GRIDLINE, linewidth=1)
    axis.set_axisbelow(True)
    axis.tick_params(axis="x", colors=TEXT_MUTED, length=0)
    axis.tick_params(axis="y", labelcolor=TEXT_SECONDARY, length=0)
    for side in ("top", "right", "bottom"):
        axis.spines[side].set_visible(False)
    axis.spines["left"].set_color(BASELINE)
figure.suptitle(
    "Forecast mean shift when one feature is shuffled, averaged over horizons",
    color=TEXT_PRIMARY,
)
plt.show()

## Checking that the ranking holds
Every score carries noise: the forecasts are sampled, and each feature is shuffled once, so a feature the model ignores still scores slightly above zero. A rerun at another `seed` draws a new shuffle and new samples. A gap between two features that keeps its sign across runs is real; one that flips is a tie. Pick the rerun seed far from the first, because a request split into batches seeds batch `i` with `seed + i`, and seeds close together share most batches. A larger `n_samples` shrinks the sampling part of the noise but not the shuffle's.

In [ ]:
RERUN_SEED = 1_000_007

rerun = client.feature_importance(
    history,
    query_times=QUERY_TIMES,
    horizons=HORIZONS,
    feature_columns=FEATURE_COLUMNS,
    target_columns=TARGET_COLUMNS,
    columns=plan.columns,
    n_samples=N_SAMPLES,
    seed=RERUN_SEED,
)
stability = pd.concat(
    {"first run": ranking, "rerun": rank_features(importance_scores(rerun))},
    axis=1,
)
stability